# ВТБ Контекст 360 — демо риск-движка «Светофор»

Ноутбук запускает симуляцию прямо в Google Colab: сравнение с жёсткими правилами, графики и подбор порогов коридоров.

**Перед запуском** замените `<ваш-логин>` в первой ячейке на свой логин GitHub. Затем выберите «Среда выполнения → Выполнить все».

> Данные синтетические, поведение жертв задано допущениями (`Assumptions`). Цифры показывают механику, а не прогноз для банка.

In [ ]:
import os, sys

REPO_URL = "https://github.com/<ваш-логин>/vtb-context360.git"

if not os.path.isdir("src/context360"):          # мы не внутри репозитория → клонируем
    if not os.path.isdir("vtb-context360"):
        !git clone -q {REPO_URL}
    os.chdir("vtb-context360")
sys.path.insert(0, os.path.abspath("src"))

from context360 import simulate, Thresholds, Corridor
from context360.report import to_text
print("Готово")

## 1. Базовый прогон: «наследие» против «Контекст 360»

In [ ]:
result = simulate(n_transfers=100_000, n_clients=10_000, seed=42)
print(to_text(result))

## 2. Ключевые метрики на графике

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 3, figsize=(14, 4))
labels = ["Наследие", "Контекст 360"]
colors = ["#9aa3ad", "#0a2896"]
sides = [result.legacy, result.new]

panels = [
    ("Ложные блоки с ручным разбором, шт.", [m.fp_manual for m in sides], "{:,.0f}"),
    ("Потери клиентов, млн ₽", [m.loss / 1e6 for m in sides], "{:.1f}"),
    ("Полнота выявления фрода, %", [m.recall * 100 for m in sides], "{:.1f}"),
]
for ax, (title, vals, fmt) in zip(axes, panels):
    bars = ax.bar(labels, vals, color=colors)
    ax.set_title(title, fontsize=11)
    ax.spines[["top", "right"]].set_visible(False)
    for b, v in zip(bars, vals):
        ax.annotate(fmt.format(v).replace(",", " "), (b.get_x() + b.get_width() / 2, v),
                    ha="center", va="bottom", fontsize=10)
plt.tight_layout()
plt.show()

## 3. Подбор порога зелёного коридора

Чем выше `green_max`, тем больше переводов проходит бесшовно. Ложных блоков становится меньше, но растёт риск пропустить фрод. Пилот на 5% трафика как раз ищет точку баланса.

In [ ]:
grid = [20, 25, 30, 35, 40, 45, 50, 55, 60]
fp, loss, green = [], [], []
for g in grid:
    r = simulate(n_transfers=60_000, n_clients=6_000, seed=7,
                 thresholds=Thresholds(green_max=g), measure_latency=False)
    fp.append(r.new.fp_manual + r.new.legit_selfservice)   # честные клиенты, которых побеспокоили
    loss.append(r.new.loss / 1e6)
    green.append(r.corridor_share(Corridor.GREEN) * 100)

fig, ax1 = plt.subplots(figsize=(9, 4.5))
ax1.plot(grid, fp, marker="o", color="#0a2896", label="Честные клиенты с доп. проверкой")
ax1.set_xlabel("Порог зелёного коридора (green_max)")
ax1.set_ylabel("Честных с проверкой, шт.", color="#0a2896")
ax2 = ax1.twinx()
ax2.plot(grid, loss, marker="s", color="#c8102e", label="Потери клиентов")
ax2.set_ylabel("Потери клиентов, млн ₽", color="#c8102e")
ax2.set_ylim(0, max(loss) * 1.3)      # ось от нуля, иначе мелкие колебания выглядят скачком
for ax in (ax1, ax2):
    ax.spines[["top"]].set_visible(False)
plt.title("Компромисс: удобство честных клиентов ↔ потери от мошенников")
plt.tight_layout()
plt.show()

for g, gr, f, l in zip(grid, green, fp, loss):
    print(f"green_max={g:>2}: зелёный {gr:5.1f}%, честных с проверкой {f:>6,}, потери {l:6.1f} млн ₽".replace(",", " "))

## 4. Свой перевод через движок

Поменяйте параметры и посмотрите, в какой коридор попадёт перевод и почему.

In [ ]:
from context360 import (Context360Engine, ClientProfile, Recipient, SessionContext,
                        Transfer, ClientBehaviour)

engine = Context360Engine()
client = ClientProfile(client_id=1, age=24, typical_limit=30_000, vulnerability_index=2, whitelist={100})
recipient = Recipient(rid=555, in_cbr_db=False, account_age_days=900, unique_senders_24h=1)
session = SessionContext(active_call=False, screen_sharing=False, phone_changed_hours=20,
                         me2me_in_24h=0, new_device=False)

t = Transfer(1, client, recipient, amount=85_000, session=session,
             behaviour=ClientBehaviour(can_qr_handshake=True))
d = engine.decide(t)
print(f"Коридор: {d.corridor.value}, score {d.score}")
for r in d.reasons:
    print("  •", r)
print("→", d.resolved_by)